In [2]:
!pip install pandas

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
def load_csvs_from_dir(directory, pattern="*.csv", concat=True, add_filename_col=True, sort=True, **read_csv_kwargs):
    """
    Load CSV files from `directory`.

    Args:
    - directory (str or Path): path to folder containing CSV files.
    - pattern (str): glob pattern to match files (default "*.csv").
    - concat (bool): if True, return a single concatenated DataFrame; if False, return a list of DataFrames.
    - add_filename_col (bool): if True, add a column '_source_file' with the filename for each row.
    - sort (bool): if True, sort files by name before loading.
    - **read_csv_kwargs: passed to `pd.read_csv` (e.g., parse_dates=["time"]).

    Returns:
    - pandas.DataFrame or list[pandas.DataFrame]
    """
    p = Path(directory)
    files = list(p.glob(pattern))
    if sort:
        files = sorted(files)
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, **read_csv_kwargs)
        except Exception as e:
            # re-raise with context
            raise RuntimeError(f"Failed to read {f}: {e}") from e
        if add_filename_col:
            df["_source_file"] = f.name
        dfs.append(df)
    if concat:
        return pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
    else:
        return dfs

In [8]:
powerwall_data_path="../minio/source-data"
df_raw = load_csvs_from_dir(powerwall_data_path, add_filename_col=False)

In [9]:
df_raw.head()

,timestamp,solar_energy_exported,consumer_energy_imported_from_grid,consumer_energy_imported_from_solar,consumer_energy_imported_from_battery
0,2020-07-31 10:00:00,0.0,0.0,0.0,0.0
1,2020-07-31 10:30:00,513.0,0.0,165.0,0.0
2,2020-07-31 11:00:00,594.0,0.0,134.0,0.0
3,2020-07-31 11:30:00,1286.0,0.0,147.0,0.0
4,2020-07-31 12:00:00,2038.0,1.0,988.0,0.0


In [6]:
df_raw.count()

timestamp\tsolar_energy_exported\tconsumer_energy_imported_from_grid\tconsumer_energy_imported_from_solar\tconsumer_energy_imported_from_battery    93944
dtype: int64

In [6]:
df_raw.head()

,timestamp,solar_energy_exported,grid_energy_imported,grid_energy_exported_from_solar,grid_energy_exported_from_battery,battery_energy_exported,battery_energy_imported_from_grid,battery_energy_imported_from_solar,consumer_energy_imported_from_grid,consumer_energy_imported_from_solar,consumer_energy_imported_from_battery,raw_timestamp,total_battery_charge,total_grid_energy_exported,total_home_usage,total_battery_discharge
0,2020-07-31 10:00:00,0.0,0.0,0.0,0.0,0,20.0,0.0,0.0,0.0,0.0,2020-07-31T10:00:00+02:00,20.0,127.0,NaN,NaN
1,2020-07-31 10:30:00,513.0,0.0,248.0,0.0,0,0.0,100.0,0.0,165.0,0.0,2020-07-31T10:30:00+02:00,100.0,248.0,NaN,NaN
2,2020-07-31 11:00:00,594.0,0.0,360.0,0.0,0,0.0,100.0,0.0,134.0,0.0,2020-07-31T11:00:00+02:00,100.0,360.0,NaN,NaN
3,2020-07-31 11:30:00,1286.0,0.0,359.0,0.0,0,0.0,780.0,0.0,147.0,0.0,2020-07-31T11:30:00+02:00,780.0,359.0,NaN,NaN
4,2020-07-31 12:00:00,2038.0,1.0,0.0,0.0,0,0.0,1050.0,1.0,988.0,0.0,2020-07-31T12:00:00+02:00,1050.0,NaN,NaN,NaN


In [7]:
df_raw.dtypes

timestamp                                 object
solar_energy_exported                    float64
grid_energy_imported                     float64
grid_energy_exported_from_solar          float64
grid_energy_exported_from_battery        float64
battery_energy_exported                    int64
battery_energy_imported_from_grid        float64
battery_energy_imported_from_solar       float64
consumer_energy_imported_from_grid       float64
consumer_energy_imported_from_solar      float64
consumer_energy_imported_from_battery    float64
raw_timestamp                             object
total_battery_charge                     float64
total_grid_energy_exported               float64
total_home_usage                         float64
total_battery_discharge                  float64
dtype: object

In [8]:
df_raw.describe()

,solar_energy_exported,grid_energy_imported,grid_energy_exported_from_solar,grid_energy_exported_from_battery,battery_energy_exported,battery_energy_imported_from_grid,battery_energy_imported_from_solar,consumer_energy_imported_from_grid,consumer_energy_imported_from_solar,consumer_energy_imported_from_battery,total_battery_charge,total_grid_energy_exported,total_home_usage,total_battery_discharge
count,93944.000000,93944.000000,93944.000000,93944.000000,93944.000000,93944.000000,93944.000000,93944.000000,93944.000000,93944.000000,7.000000,18170.000000,93916.000000,34418.000000
mean,383.918845,94.772001,143.225435,0.098160,91.879002,1.228932,106.481811,93.543282,134.211599,91.780842,602.857143,254.571793,319.576877,186.462897
std,599.836814,254.625798,410.471682,0.298193,178.493410,13.142307,284.639951,252.724329,278.023788,178.459799,587.670948,501.162681,376.829695,198.155438
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,20.000000,0.000417,16.000000,1.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,100.000000,1.000000,117.000000,100.000000
50%,0.503930,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.011319,0.000000,570.000000,1.000000,151.000000,130.000000
75%,613.036053,8.000000,0.608585,0.000000,122.000000,0.000000,0.000000,8.000000,141.500000,121.000000,915.000000,195.224744,351.000000,160.000000
max,2600.279121,3024.500000,2456.356447,3.000000,2170.000000,850.000000,2170.000000,3024.500000,2200.097091,2170.000000,1600.000000,2293.000000,3542.000000,2170.000000


In [9]:
df=pd.DataFrame()
timestamps = pd.to_datetime(df_raw["timestamp"], dayfirst=False, errors="raise")
df["timestamp"]=timestamps
df["time"]=timestamps.dt.time
df["month"]=timestamps.dt.month

In [10]:
df.head()

,timestamp,time,month
0,2020-07-31 10:00:00,10:00:00,7
1,2020-07-31 10:30:00,10:30:00,7
2,2020-07-31 11:00:00,11:00:00,7
3,2020-07-31 11:30:00,11:30:00,7
4,2020-07-31 12:00:00,12:00:00,7


In [11]:
df.dtypes

timestamp    datetime64[ns]
time                 object
month                 int32
dtype: object

In [12]:
df['solar_energy_exported'] = df_raw['solar_energy_exported']
df['total_home_usage']=df_raw['consumer_energy_imported_from_grid']+df_raw['consumer_energy_imported_from_solar']+df_raw['consumer_energy_imported_from_battery']

In [13]:
df.count()

timestamp                93944
time                     93944
month                    93944
solar_energy_exported    93944
total_home_usage         93944
dtype: int64

In [14]:
# Remove hours at early morning or late evening where no solar production can occur
df[ ( (df.time.astype(str)>='06:00:00') & (df.time.astype(str)<'21:00:00') )].describe()

,timestamp,month,solar_energy_exported,total_home_usage
count,58716,58716.000000,58716.000000,58716.000000
mean,2023-04-05 22:03:00.441446912,6.732441,614.236790,360.713805
min,2020-07-31 10:00:00,1.000000,0.000000,0.000000
25%,2021-12-02 14:22:30,4.000000,18.000000,127.000000
50%,2023-04-05 18:45:00,7.000000,337.346581,164.000000
75%,2024-08-07 08:07:30,10.000000,1134.137314,452.000000
max,2025-12-09 12:30:00,12.000000,2600.279121,3542.000000
std,NaN,3.436376,658.951466,412.152598


In [15]:
df = df[ ( (df.time.astype(str)>='06:00:00') & (df.time.astype(str)<'21:00:00') )]

In [16]:
df.count()

timestamp                58716
time                     58716
month                    58716
solar_energy_exported    58716
total_home_usage         58716
dtype: int64

In [17]:
# compute solar surplus (at minimum it must be 0)
df['solar_surplus']=df['solar_energy_exported'] - df['total_home_usage']
df.solar_surplus=df.solar_surplus.apply(lambda x: max(0,x))
df.head()

,timestamp,time,month,solar_energy_exported,total_home_usage,solar_surplus
0,2020-07-31 10:00:00,10:00:00,7,0.0,0.0,0.0
1,2020-07-31 10:30:00,10:30:00,7,513.0,165.0,348.0
2,2020-07-31 11:00:00,11:00:00,7,594.0,134.0,460.0
3,2020-07-31 11:30:00,11:30:00,7,1286.0,147.0,1139.0
4,2020-07-31 12:00:00,12:00:00,7,2038.0,989.0,1049.0


In [18]:
df.describe()

,timestamp,month,solar_energy_exported,total_home_usage,solar_surplus
count,58716,58716.000000,58716.000000,58716.000000,58716.000000
mean,2023-04-05 22:03:00.441446912,6.732441,614.236790,360.713805,396.331347
min,2020-07-31 10:00:00,1.000000,0.000000,0.000000,0.000000
25%,2021-12-02 14:22:30,4.000000,18.000000,127.000000,0.000000
50%,2023-04-05 18:45:00,7.000000,337.346581,164.000000,46.000000
75%,2024-08-07 08:07:30,10.000000,1134.137314,452.000000,694.000000
max,2025-12-09 12:30:00,12.000000,2600.279121,3542.000000,2456.356447
std,NaN,3.436376,658.951466,412.152598,551.481531


In [19]:
# check null values
df.dropna().count()

timestamp                58716
time                     58716
month                    58716
solar_energy_exported    58716
total_home_usage         58716
solar_surplus            58716
dtype: int64

In [20]:
df['solar_surplus'].describe()

count    58716.000000
mean       396.331347
std        551.481531
min          0.000000
25%          0.000000
50%         46.000000
75%        694.000000
max       2456.356447
Name: solar_surplus, dtype: float64

In [21]:
min(df.time.astype(str))

'06:00:00'

In [22]:
df[ df.time.astype(str)=='06:00:00'].describe()

,timestamp,month,solar_energy_exported,total_home_usage,solar_surplus
count,1957,1957.000000,1957.000000,1957.000000,1957.000000
mean,2023-04-06 06:00:00.000000256,6.733776,6.908169,220.925485,0.026968
min,2020-08-01 06:00:00,1.000000,0.000000,74.494347,0.000000
25%,2021-12-03 06:00:00,4.000000,0.000000,104.500000,0.000000
50%,2023-04-06 06:00:00,7.000000,0.000000,140.500000,0.000000
75%,2024-08-07 06:00:00,10.000000,0.000000,211.895232,0.000000
max,2025-12-09 06:00:00,12.000000,124.790451,1601.500000,30.590903
std,NaN,3.438497,17.167663,210.838503,0.833013


In [23]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests

In [24]:
import pandas as pd
import requests_cache
from retry_requests import retry
import openmeteo_requests
import weather_data as wd

In [25]:
# def get_historical_weather(latitude: float, longitude: float, start_date: str, end_date=None, timezone="Europe/Berlin"):
    
#     # Setup the Open-Meteo API client with cache and retry on error
#     cache_session = requests_cache.CachedSession('.cache', expire_after = -1)
#     retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
#     openmeteo = openmeteo_requests.Client(session = retry_session)
    
#     # Make sure all required weather variables are listed here
#     # The order of variables in hourly or daily is important to assign them correctly below
#     url = "https://archive-api.open-meteo.com/v1/archive"
#     params = {
#     	"latitude": {latitude},
#     	"longitude": {longitude},
#     	"start_date": {start_date},
#     	"end_date": { (end_date if end_date is not None else start_date) },
#     	#"daily": ["sunrise", "sunset"],
#     	"hourly": ["weather_code", "cloud_cover", "temperature"],
#     	"timezone": {timezone},
#     }
#     responses = openmeteo.weather_api(url, params=params)
    
#     # Process first location. Add a for-loop for multiple locations or weather models
#     response = responses[0]
    
#     # Process hourly data. The order of variables needs to be the same as requested.
#     hourly = response.Hourly()
#     #print(response)
#     hourly_weather_code = hourly.Variables(0).ValuesAsNumpy()
#     hourly_cloud_cover = hourly.Variables(1).ValuesAsNumpy()
#     hourly_temperature = hourly.Variables(2).ValuesAsNumpy()
    
#     hourly_data = {"timestamp": pd.date_range(
#     	start = pd.to_datetime(hourly.Time(), unit = "s", utc = False),
#     	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = False),
#     	freq = pd.Timedelta(seconds = hourly.Interval()),
#     	inclusive = "left"
#     )}
    
#     hourly_data["weather_code"] = hourly_weather_code
#     hourly_data["cloud_cover"] = hourly_cloud_cover
#     hourly_data["temperature"] = hourly_temperature
    
#     hourly_df = pd.DataFrame(data = hourly_data)
#     #print("\nHourly data\n", hourly_dataframe)

#     #ts = pd.to_datetime(hourly_df["date_time"], dayfirst=False, errors="raise")
#     # hourly_df["date"]=ts.dt.date
#     # hourly_df["time"]=ts.dt.time
    
#     return hourly_df
    
#     # Process daily data. The order of variables needs to be the same as requested.
#     #daily = response.Daily()
#     #daily_sunrise = daily.Variables(0).ValuesInt64AsNumpy()
#     #daily_sunset = daily.Variables(1).ValuesInt64AsNumpy()
    
#     #daily_data = {"date": pd.date_range(
#     # 	start = pd.to_datetime(daily.Time(), unit = "s", utc = True),
#     #	end =  pd.to_datetime(daily.TimeEnd(), unit = "s", utc = True),
#     #	freq = pd.Timedelta(seconds = daily.Interval()),
#     #	inclusive = "left"
#     #)}
    
#     #daily_data["sunrise"] = daily_sunrise
#     #daily_data["sunset"] = daily_sunset
    
#     #daily_dataframe = pd.DataFrame(data = daily_data)
#     #print("\nDaily data\n", daily_dataframe)
    

In [26]:
# Solar plant Coordinates
latitude=45.704178167781265
longitude=9.663174137924273
timezone="Europe/Berlin"
stard_date='2026-02-09'
hourly_df = wd.get_weather_data(stard_date,stard_date,latitude, longitude, timezone)

In [27]:
print("\nHourly data\n", hourly_df)


Hourly data
              timestamp  weather_code  cloud_cover  temperature
0  2026-02-08 23:00:00           3.0         22.0       5.8685
1  2026-02-09 00:00:00           3.0         42.0       5.4685
2  2026-02-09 01:00:00           3.0         98.0       5.4685
3  2026-02-09 02:00:00           3.0         48.0       5.3685
4  2026-02-09 03:00:00           3.0         11.0       5.3685
5  2026-02-09 04:00:00           3.0         58.0       5.9685
6  2026-02-09 05:00:00           3.0          0.0       6.0185
7  2026-02-09 06:00:00           3.0          0.0       5.8185
8  2026-02-09 07:00:00           3.0          9.0       6.2685
9  2026-02-09 08:00:00           3.0          0.0       7.0185
10 2026-02-09 09:00:00           3.0          0.0       8.3185
11 2026-02-09 10:00:00          61.0          0.0       9.0185
12 2026-02-09 11:00:00          80.0          0.0       9.4185
13 2026-02-09 12:00:00           3.0         17.0      10.2685
14 2026-02-09 13:00:00           3.0     

In [28]:
hourly_df.cloud_cover.describe()

count     24.000000
mean      30.083334
std       33.053207
min        0.000000
25%        3.000000
50%       17.500000
75%       43.500000
max      100.000000
Name: cloud_cover, dtype: float64

In [29]:
df.head()

,timestamp,time,month,solar_energy_exported,total_home_usage,solar_surplus
0,2020-07-31 10:00:00,10:00:00,7,0.0,0.0,0.0
1,2020-07-31 10:30:00,10:30:00,7,513.0,165.0,348.0
2,2020-07-31 11:00:00,11:00:00,7,594.0,134.0,460.0
3,2020-07-31 11:30:00,11:30:00,7,1286.0,147.0,1139.0
4,2020-07-31 12:00:00,12:00:00,7,2038.0,989.0,1049.0


In [30]:
df.dtypes

timestamp                datetime64[ns]
time                             object
month                             int32
solar_energy_exported           float64
total_home_usage                float64
solar_surplus                   float64
dtype: object

In [31]:
print(df.timestamp.iloc(0)[0].date())

2020-07-31


In [32]:
# Get min and max dates
min_date=min(df.timestamp).date()
max_date=max(df.timestamp).date()
print(min_date, max_date)

2020-07-31 2025-12-09


In [33]:
# Get historiscal weather forecast by hour
from pathlib import Path
weather_filename = "weather_{min_date}_to_{max_date}.csv".format(min_date=min_date, max_date=max_date)
weather_file = Path("./{weather_filename}".format(weather_filename=weather_filename))
hourly_df = None
if weather_file.is_file():
    print("File '{weather_filename}' found, lodaing data".format(weather_filename=weather_filename))
    hourly_df = pd.read_csv(weather_filename, sep='\t', encoding='utf-8')
else:
    print("File '{weather_filename}' NOT found, calling meteo data API".format(weather_filename=weather_filename))
    hourly_df = get_historical_weather(latitude, longitude, min_date, max_date)

File 'weather_2020-07-31_to_2025-12-09.csv' found, lodaing data


In [34]:
hourly_df.head()

,timestamp,weather_code,cloud_cover,temperature
0,2020-07-30 23:00:00,0.0,19.0,21.35
1,2020-07-31 00:00:00,0.0,11.0,22.25
2,2020-07-31 01:00:00,3.0,99.0,22.45
3,2020-07-31 02:00:00,51.0,99.0,21.60
4,2020-07-31 03:00:00,3.0,94.0,19.75


In [35]:
hourly_df.dtypes

timestamp        object
weather_code    float64
cloud_cover     float64
temperature     float64
dtype: object

In [36]:
df.dtypes

timestamp                datetime64[ns]
time                             object
month                             int32
solar_energy_exported           float64
total_home_usage                float64
solar_surplus                   float64
dtype: object

In [37]:
# Save the weather historical data
if not weather_file.is_file():
    #now=pd.Timestamp.now().strftime('%Y-%m-%d_%H%M%S')
    #filename = "{now}_weather_{min_date}_to_{max_date}.csv".format(now=now, min_date=min_date, max_date=max_date)
    hourly_df.to_csv(weather_filename, sep='\t', encoding='utf-8', index=False, header=True)
    print('Saved weather data to {filename}'.format(filename=weather_filename))

In [38]:
# merge energy data with weather historical data
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=False)
hourly_df['timestamp'] = pd.to_datetime(hourly_df['timestamp'], utc=False)
df_merge=pd.merge(df, hourly_df, how='left', on='timestamp')

In [39]:
df_merge.count()

timestamp                58716
time                     58716
month                    58716
solar_energy_exported    58716
total_home_usage         58716
solar_surplus            58716
weather_code             29358
cloud_cover              29358
temperature              29358
dtype: int64

In [40]:
df_merge.head()

,timestamp,time,month,solar_energy_exported,total_home_usage,solar_surplus,weather_code,cloud_cover,temperature
0,2020-07-31 10:00:00,10:00:00,7,0.0,0.0,0.0,0.0,14.0,28.85
1,2020-07-31 10:30:00,10:30:00,7,513.0,165.0,348.0,NaN,NaN,NaN
2,2020-07-31 11:00:00,11:00:00,7,594.0,134.0,460.0,0.0,15.0,29.70
3,2020-07-31 11:30:00,11:30:00,7,1286.0,147.0,1139.0,NaN,NaN,NaN
4,2020-07-31 12:00:00,12:00:00,7,2038.0,989.0,1049.0,1.0,30.0,30.65


In [41]:
# Fill missing data at half-hours with that from the beginning of the corresponding hours
import numpy as np

df_missing = df_merge[ ( np.isnan(df_merge.weather_code) | np.isnan(df_merge.cloud_cover) | np.isnan(df_merge.temperature) ) ]

if df_missing.size>0:
    print('Filling missing weather data at half-hours...')
    df_merge = df_merge.reset_index()  # make sure indexes pair with number of rows
    
    for idx, row in df_merge.iterrows():    
        if row['timestamp'].minute==30:
            source_idx = ( idx-1 if idx > 0 else idx+1 )
            if (np.isnan(row['weather_code'])):
                df_merge.at[ idx, 'weather_code'] = df_merge.loc[source_idx]['weather_code']
            if (np.isnan(row['cloud_cover'])):
                df_merge.at[ idx, 'cloud_cover'] = df_merge.loc[source_idx]['cloud_cover']
            if (np.isnan(row['temperature'])):
                df_merge.at[ idx, 'temperature'] = df_merge.loc[source_idx]['temperature']

Filling missing weather data at half-hours...


In [42]:
df_merge.head(20)

,index,timestamp,time,month,solar_energy_exported,total_home_usage,solar_surplus,weather_code,cloud_cover,temperature
0,0,2020-07-31 10:00:00,10:00:00,7,0.0,0.0,0.0,0.0,14.0,28.85
1,1,2020-07-31 10:30:00,10:30:00,7,513.0,165.0,348.0,0.0,14.0,28.85
2,2,2020-07-31 11:00:00,11:00:00,7,594.0,134.0,460.0,0.0,15.0,29.70
3,3,2020-07-31 11:30:00,11:30:00,7,1286.0,147.0,1139.0,0.0,15.0,29.70
4,4,2020-07-31 12:00:00,12:00:00,7,2038.0,989.0,1049.0,1.0,30.0,30.65
5,5,2020-07-31 12:30:00,12:30:00,7,2052.0,451.0,1601.0,1.0,30.0,30.65
6,6,2020-07-31 13:00:00,13:00:00,7,2081.0,162.0,1919.0,2.0,69.0,31.10
7,7,2020-07-31 13:30:00,13:30:00,7,2031.0,144.0,1887.0,2.0,69.0,31.10
8,8,2020-07-31 14:00:00,14:00:00,7,1994.0,179.0,1815.0,2.0,68.0,30.80
9,9,2020-07-31 14:30:00,14:30:00,7,1913.0,160.0,1753.0,2.0,68.0,30.80


In [43]:
# Check whether some weather data is missing
df_merge[ ( np.isnan(df_merge['weather_code']) | np.isnan(df_merge['cloud_cover']) | np.isnan(df_merge['temperature']) )].size

0

In [44]:
df_merge

,index,timestamp,time,month,solar_energy_exported,total_home_usage,solar_surplus,weather_code,cloud_cover,temperature
0,0,2020-07-31 10:00:00,10:00:00,7,0.0,0.0,0.0,0.0,14.0,28.85
1,1,2020-07-31 10:30:00,10:30:00,7,513.0,165.0,348.0,0.0,14.0,28.85
2,2,2020-07-31 11:00:00,11:00:00,7,594.0,134.0,460.0,0.0,15.0,29.70
3,3,2020-07-31 11:30:00,11:30:00,7,1286.0,147.0,1139.0,0.0,15.0,29.70
4,4,2020-07-31 12:00:00,12:00:00,7,2038.0,989.0,1049.0,1.0,30.0,30.65
...,...,...,...,...,...,...,...,...,...,...
58711,58711,2025-12-09 10:30:00,10:30:00,12,1216.0,870.0,346.0,0.0,0.0,10.20
58712,58712,2025-12-09 11:00:00,11:00:00,12,1288.0,184.0,1104.0,0.0,0.0,11.20
58713,58713,2025-12-09 11:30:00,11:30:00,12,1340.0,1410.0,0.0,0.0,0.0,11.20
58714,58714,2025-12-09 12:00:00,12:00:00,12,1332.0,1399.0,0.0,0.0,0.0,11.95


In [45]:
# Set features
categorical=['time', 'month','weather_code', 'cloud_cover']
numerical=[ 'temperature' ]

In [46]:
df_merge[categorical] = df_merge[categorical].astype(str)

In [47]:
df_merge.dtypes
df_merge[categorical + numerical].iloc[:10]

,time,month,weather_code,cloud_cover,temperature
0,10:00:00,7,0.0,14.0,28.85
1,10:30:00,7,0.0,14.0,28.85
2,11:00:00,7,0.0,15.0,29.70
3,11:30:00,7,0.0,15.0,29.70
4,12:00:00,7,1.0,30.0,30.65
5,12:30:00,7,1.0,30.0,30.65
6,13:00:00,7,2.0,69.0,31.10
7,13:30:00,7,2.0,69.0,31.10
8,14:00:00,7,2.0,68.0,30.80
9,14:30:00,7,2.0,68.0,30.80


In [48]:
!pip install scikit-learn

In [49]:
df_merge.count()

index                    58716
timestamp                58716
time                     58716
month                    58716
solar_energy_exported    58716
total_home_usage         58716
solar_surplus            58716
weather_code             58716
cloud_cover              58716
temperature              58716
dtype: int64

In [50]:
# Categorical features must be converter in numerical ones by using Vectorizers

In [51]:
# Split merged data set 75% train / 25% test
X_Train_df = df_merge.sample(frac = 0.75)
X_Test_df = df_merge.drop(X_Train_df.index)

In [52]:
X_Train_df.head()

,index,timestamp,time,month,solar_energy_exported,total_home_usage,solar_surplus,weather_code,cloud_cover,temperature
9789,9789,2021-06-22 14:30:00,14:30:00,6,1654.466552,504.804066,1149.662486,51.0,73.0,25.80
13035,13035,2021-10-08 17:30:00,17:30:00,10,260.556739,642.517968,0.000000,2.0,72.0,15.10
49032,49032,2025-01-20 16:00:00,16:00:00,1,40.000000,567.500000,0.000000,3.0,100.0,5.60
22152,22152,2022-08-08 16:00:00,16:00:00,8,1574.387496,106.249977,1468.137519,2.0,77.0,28.60
57563,57563,2025-11-01 06:30:00,06:30:00,11,0.000000,201.000000,0.000000,3.0,100.0,13.05


In [53]:
X_Train_df.count()

index                    44037
timestamp                44037
time                     44037
month                    44037
solar_energy_exported    44037
total_home_usage         44037
solar_surplus            44037
weather_code             44037
cloud_cover              44037
temperature              44037
dtype: int64

In [54]:
X_Test_df.head()

,index,timestamp,time,month,solar_energy_exported,total_home_usage,solar_surplus,weather_code,cloud_cover,temperature
2,2,2020-07-31 11:00:00,11:00:00,7,594.0,134.0,460.0,0.0,15.0,29.70
5,5,2020-07-31 12:30:00,12:30:00,7,2052.0,451.0,1601.0,1.0,30.0,30.65
20,20,2020-07-31 20:00:00,20:00:00,7,79.0,138.0,0.0,0.0,15.0,26.15
23,23,2020-08-01 06:30:00,06:30:00,8,50.0,139.0,0.0,0.0,0.0,24.05
40,40,2020-08-01 15:00:00,15:00:00,8,1735.0,629.0,1106.0,1.0,29.0,30.30


In [55]:
X_Test_df.count()

index                    14679
timestamp                14679
time                     14679
month                    14679
solar_energy_exported    14679
total_home_usage         14679
solar_surplus            14679
weather_code             14679
cloud_cover              14679
temperature              14679
dtype: int64

In [56]:
from sklearn.feature_extraction import DictVectorizer

In [57]:
X_Train_df[ categorical + numerical ].head()

,time,month,weather_code,cloud_cover,temperature
9789,14:30:00,6,51.0,73.0,25.80
13035,17:30:00,10,2.0,72.0,15.10
49032,16:00:00,1,3.0,100.0,5.60
22152,16:00:00,8,2.0,77.0,28.60
57563,06:30:00,11,3.0,100.0,13.05


In [58]:
# Get dictionaries of categorical features
X_Train_dicts = X_Train_df[categorical+numerical].to_dict(orient='records')

In [59]:
X_Train_dicts[:10]

[{'time': '14:30:00',
  'month': '6',
  'weather_code': '51.0',
  'cloud_cover': '73.0',
  'temperature': 25.8},
 {'time': '17:30:00',
  'month': '10',
  'weather_code': '2.0',
  'cloud_cover': '72.0',
  'temperature': 15.1},
 {'time': '16:00:00',
  'month': '1',
  'weather_code': '3.0',
  'cloud_cover': '100.0',
  'temperature': 5.6},
 {'time': '16:00:00',
  'month': '8',
  'weather_code': '2.0',
  'cloud_cover': '77.0',
  'temperature': 28.6},
 {'time': '06:30:00',
  'month': '11',
  'weather_code': '3.0',
  'cloud_cover': '100.0',
  'temperature': 13.05},
 {'time': '17:30:00',
  'month': '6',
  'weather_code': '3.0',
  'cloud_cover': '100.0',
  'temperature': 25.35},
 {'time': '09:30:00',
  'month': '3',
  'weather_code': '0.0',
  'cloud_cover': '0.0',
  'temperature': 15.75},
 {'time': '11:00:00',
  'month': '3',
  'weather_code': '51.0',
  'cloud_cover': '100.0',
  'temperature': 8.25},
 {'time': '20:30:00',
  'month': '1',
  'weather_code': '3.0',
  'cloud_cover': '100.0',
  'tem

In [62]:
# Convert categorical features into a dataset which has as many columns as the label of each category
dv = DictVectorizer()
X_train = dv.fit_transform(X_Train_dicts)

In [63]:
X_train

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 220185 stored elements and shape (44037, 157)>

In [64]:
# Same approach for test data set
X_Test_dicts = X_Test_df[categorical+numerical].to_dict(orient='records')

In [65]:
X_Test_dicts[:10]

[{'time': '11:00:00',
  'month': '7',
  'weather_code': '0.0',
  'cloud_cover': '15.0',
  'temperature': 29.7},
 {'time': '12:30:00',
  'month': '7',
  'weather_code': '1.0',
  'cloud_cover': '30.0',
  'temperature': 30.65},
 {'time': '20:00:00',
  'month': '7',
  'weather_code': '0.0',
  'cloud_cover': '15.0',
  'temperature': 26.15},
 {'time': '06:30:00',
  'month': '8',
  'weather_code': '0.0',
  'cloud_cover': '0.0',
  'temperature': 24.05},
 {'time': '15:00:00',
  'month': '8',
  'weather_code': '1.0',
  'cloud_cover': '29.0',
  'temperature': 30.3},
 {'time': '15:30:00',
  'month': '8',
  'weather_code': '1.0',
  'cloud_cover': '29.0',
  'temperature': 30.3},
 {'time': '17:00:00',
  'month': '8',
  'weather_code': '1.0',
  'cloud_cover': '39.0',
  'temperature': 30.25},
 {'time': '19:00:00',
  'month': '8',
  'weather_code': '3.0',
  'cloud_cover': '86.0',
  'temperature': 26.05},
 {'time': '19:30:00',
  'month': '8',
  'weather_code': '3.0',
  'cloud_cover': '86.0',
  'temperatu

In [66]:
X_test = dv.transform(X_Test_dicts)

In [67]:
X_test

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 73395 stored elements and shape (14679, 157)>

In [68]:
target = 'solar_surplus'
y_train = X_Train_df[target].values
y_test = X_Test_df[target].values

In [69]:
y_train.size

44037

In [70]:
y_test.size

14679

In [71]:
# Save data file
source_data_filename='data/source_data.csv'
df_merge.to_csv(source_data_filename, sep='\t', encoding='utf-8', index=False, header=True)

In [72]:
!pip install mlflow

In [73]:
import mlflow
from sklearn.metrics import root_mean_squared_error # standard deviation = sqrt(mean_squared_error)

experiment_name="solar-surplus-experiment-with-temperature"
mlflow.set_tracking_uri("sqlite:///mlflow.db") # to open mlflow UI:  mlflow ui --backend-store-uri="sqlite:///mlflow.db"
mlflow.set_experiment(experiment_name)
model_path="models"

2026/02/09 16:51:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/02/09 16:51:48 INFO mlflow.store.db.utils: Updating database tables
2026/02/09 16:51:48 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/09 16:51:48 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/09 16:51:48 INFO alembic.runtime.migration: Running upgrade bf29a5ff90ea -> 1bd49d398cd23, add secrets tables
2026/02/09 16:51:48 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/09 16:51:48 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2026/02/09 16:51:48 INFO mlflow.tracking.fluent: Experiment with name 'solar-surplus-experiment-with-temperature' does not exist. Creating a new experiment.


In [74]:
!pip install xgboost hyperopt

In [75]:
# hyperparameter tuning
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials # use hyperpot package to perform parameter tuning
from hyperopt.pyll import scope

/home/calinet/Documents/Code/anaconda3/envs/solar-energy-surplus-estimator/lib/python3.13/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [76]:
X_Train_dm = xgb.DMatrix(X_train, label=y_train)
X_test_dm = xgb.DMatrix(X_test, label=y_test)

In [77]:
def objective(params):
    mlflow.set_experiment(experiment_name)
    with mlflow.start_run():

        model_type="xgboost"
        model_name=experiment_name+"_"+model_type
        mlflow.set_tag("model_type", model_type)
        mlflow.log_params(params)

        mlflow.xgboost.autolog()
        booster = xgb.train(
            params=params,
            dtrain=X_Train_dm,
            num_boost_round=1000, # Maximum number of iterations for hyperparameter tuning
            evals=[(X_test_dm, 'validation')],
            early_stopping_rounds=50 # Stop after 50 iterations with no improvement
        )
        y_pred = booster.predict(X_test_dm)
        rmse = root_mean_squared_error(y_test, y_pred)
        mlflow.log_metric("rmse_test", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [78]:
# tuning parameters (see https://hyperopt.github.io/hyperopt/ for reference )
search_space = {
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),  # explore solutions from exp(-3) and exp(0)
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',
    'seed': 42
}

In [80]:
best_result = fmin(
    fn=objective,
    space=search_space,
    algo=tpe.suggest,
    max_evals=20,
    trials=Trials()
)

[0]	validation-rmse:486.25483                         
[1]	validation-rmse:440.21640                         
[2]	validation-rmse:410.97946                         
[3]	validation-rmse:394.08761                         
[4]	validation-rmse:385.66966                         
[5]	validation-rmse:382.69785                         
[6]	validation-rmse:383.07224                         
[7]	validation-rmse:385.22368                         
[8]	validation-rmse:388.26294                         
[9]	validation-rmse:391.60689                         
[10]	validation-rmse:394.87305                        
[11]	validation-rmse:397.89832                        
[12]	validation-rmse:400.61251                        
[13]	validation-rmse:403.02846                        
[14]	validation-rmse:405.13493                        
[15]	validation-rmse:406.90918                        
[16]	validation-rmse:408.41456                        
[17]	validation-rmse:409.68514                        
[18]	valid

2026/02/09 16:53:19 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.

2026/02/09 16:53:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



[0]	validation-rmse:464.31113                                                   
[1]	validation-rmse:410.78898                                                   
[2]	validation-rmse:379.79179                                                   
[3]	validation-rmse:364.08244                                                   
[4]	validation-rmse:357.22008                                                   
[5]	validation-rmse:354.73285                                                   
[6]	validation-rmse:354.33534                                                   
[7]	validation-rmse:354.97206                                                   
[8]	validation-rmse:356.27369                                                   
[9]	validation-rmse:358.16804                                                   
[10]	validation-rmse:359.38693                                                  
[11]	validation-rmse:360.91985                                                  
[12]	validation-rmse:362.283

2026/02/09 16:53:32 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.

2026/02/09 16:53:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



[0]	validation-rmse:507.86859                                                   
[1]	validation-rmse:471.70369                                                  
[2]	validation-rmse:443.18539                                                  
[3]	validation-rmse:421.17105                                                  
[4]	validation-rmse:404.91857                                                  
[5]	validation-rmse:393.07791                                                  
[6]	validation-rmse:384.81015                                                  
[7]	validation-rmse:379.59500                                                  
[8]	validation-rmse:376.51914                                                  
[9]	validation-rmse:375.09258                                                  
[10]	validation-rmse:374.86010                                                 
[11]	validation-rmse:375.40461                                                 
[12]	validation-rmse:376.51550         

2026/02/09 16:53:57 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.

2026/02/09 16:53:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



... (31 outputs left out)

2026/02/09 16:58:19 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.

2026/02/09 16:58:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



[0]	validation-rmse:397.87787                                                  
[1]	validation-rmse:382.70768                                                  
[2]	validation-rmse:393.31105                                                  
[3]	validation-rmse:401.65609                                                  
[4]	validation-rmse:406.45344                                                  
[5]	validation-rmse:408.90109                                                  
[6]	validation-rmse:410.10524                                                  
[7]	validation-rmse:410.68659                                                  
[8]	validation-rmse:410.92954                                                  
[9]	validation-rmse:411.04373                                                  
[10]	validation-rmse:411.09676                                                 
[11]	validation-rmse:411.11897                                                 
[12]	validation-rmse:411.12947          

2026/02/09 16:58:36 WARNING mlflow.xgboost: Failed to infer model signature: could not sample data to infer model signature: please ensure that autologging is enabled before constructing the dataset.

2026/02/09 16:58:36 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



100%|██████████| 20/20 [05:38<00:00, 16.95s/trial, best loss: 314.314452647814]


In [85]:
# Get the best run
df_runs = mlflow.search_runs(filter_string="status='FINISHED'")
df_best_run=df_runs.loc[ df_runs['metrics.rmse_test'].idxmin() ]

In [86]:
df_best_run

run_id                                           ab2f9848959646eeb7a602180955b292
experiment_id                                                                   1
status                                                                   FINISHED
artifact_uri                    /home/calinet/Documents/Code/solar-energy-surp...
start_time                                       2026-02-09 15:56:52.749000+00:00
end_time                                         2026-02-09 15:57:24.296000+00:00
metrics.rmse_test                                                      314.314453
metrics.best_iteration                                                      703.0
metrics.validation-rmse                                                 314.04444
metrics.stopped_iteration                                                   753.0
params.early_stopping_rounds                                                   50
params.maximize                                                              None
params.verbose_e

In [87]:
print(f'best rmse_test: {df_best_run['metrics.rmse_test']}')
best_run_id = df_best_run['run_id']
print(f'best run_id: {best_run_id}')

best rmse_test: 314.314452647814
best run_id: ab2f9848959646eeb7a602180955b292


In [131]:
import pickle

In [132]:

mlflow.xgboost.autolog(disable=True)
model_type="xgboost"
model_name=experiment_name+"_"+model_type
mlflow.set_tag("model_type", model_type)


In [134]:
mlflow.end_run()
with mlflow.start_run():
    
    # train = xgb.DMatrix(X_train, label=y_train)
    # valid = xgb.DMatrix(X_val, label=y_val)

    X_Train_dm = xgb.DMatrix(X_train, label=y_train)
    X_test_dm = xgb.DMatrix(X_test, label=y_test)

    best_params = {
        "learning_rate": 0.06442636015018084,
        "max_depth": 17,
        "min_child_weight": 11.057750768573072,
        "objective": "reg:squarederror",
        "reg_alpha": 0.07238551460828992,
        "reg_lambda": 0.06289038778129306,
        "seed": 42
    }


    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=X_Train_dm,
        num_boost_round=1000,
        evals=[(X_test_dm, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(X_test_dm)
    rmse = root_mean_squared_error(y_test, y_pred)
    mlflow.log_metric("rmse_test", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="solar-energy-surplus-predictor")

[0]	validation-rmse:526.33157
[1]	validation-rmse:507.54025
[2]	validation-rmse:490.42072
[3]	validation-rmse:474.94036
[4]	validation-rmse:460.73355
[5]	validation-rmse:447.88637
[6]	validation-rmse:436.06317
[7]	validation-rmse:425.29676
[8]	validation-rmse:415.13720
[9]	validation-rmse:406.16982
[10]	validation-rmse:398.02563
[11]	validation-rmse:390.53729
[12]	validation-rmse:383.60830
[13]	validation-rmse:377.52368
[14]	validation-rmse:372.15543
[15]	validation-rmse:367.17539
[16]	validation-rmse:362.79900
[17]	validation-rmse:358.58856
[18]	validation-rmse:354.93003
[19]	validation-rmse:351.46212
[20]	validation-rmse:348.46184
[21]	validation-rmse:345.67453
[22]	validation-rmse:343.20029
[23]	validation-rmse:340.98020
[24]	validation-rmse:339.04588
[25]	validation-rmse:337.22364
[26]	validation-rmse:335.49306
[27]	validation-rmse:334.08306
[28]	validation-rmse:332.84295
[29]	validation-rmse:331.62127
[30]	validation-rmse:330.51096
[31]	validation-rmse:329.25791
[32]	validation-rm

2026/01/14 15:51:04 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
